# Interactive Sigma Control (Lecture 10)
Move the four points with sliders and vary covariance Sigma for Euclidean vs Mahalanobis comparison.


In [ ]:
import math
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

DEFAULT_POINTS = [
    [0.0, 0.0],
    [1.0, 1.0],
    [2.0, 0.5],
    [-1.0, 1.6],
]


def eig2(a, b, c):
    tr = a + c
    det = a * c - b * b
    disc = math.sqrt(max(tr * tr - 4 * det, 0.0))
    return (tr + disc) / 2.0, (tr - disc) / 2.0


point_boxes = []
point_widgets = []
for idx, (xv, yv) in enumerate(DEFAULT_POINTS, start=1):
    xw = widgets.FloatSlider(value=xv, min=-4.0, max=4.0, step=0.1, description='x', readout_format='.1f', continuous_update=False)
    yw = widgets.FloatSlider(value=yv, min=-4.0, max=4.0, step=0.1, description='y', readout_format='.1f', continuous_update=False)
    point_widgets.append((xw, yw))
    point_boxes.append(
        widgets.VBox(
            [widgets.HTML(value=f'<b>x{idx}</b>'), xw, yw],
            layout=widgets.Layout(border='1px solid #cbd5e1', padding='10px', border_radius='12px', width='250px')
        )
    )

sx = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='sigma_x', readout_format='.1f', continuous_update=False)
sy = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='sigma_y', readout_format='.1f', continuous_update=False)
rho = widgets.FloatSlider(value=0.0, min=-0.95, max=0.95, step=0.05, description='rho', readout_format='.2f', continuous_update=False)
reset_btn = widgets.Button(description='Reset default points')
out = widgets.Output()


def current_points():
    return [[float(xw.value), float(yw.value)] for xw, yw in point_widgets]


def render(*_):
    pts = current_points()
    cov = np.array([
        [sx.value * sx.value, rho.value * sx.value * sy.value],
        [rho.value * sx.value * sy.value, sy.value * sy.value],
    ], dtype=float)
    det = float(np.linalg.det(cov))
    if abs(det) < 1e-9:
        det = 1e-9
    inv = np.linalg.inv(cov + 1e-9 * np.eye(2))

    de = []
    dm = []
    for px, py in pts:
        vec = np.array([px, py], dtype=float)
        de.append(float(np.linalg.norm(vec)))
        dm.append(float(math.sqrt(vec @ inv @ vec)))

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    t = np.linspace(0.0, 2.0 * math.pi, 240)
    l1, l2 = eig2(cov[0, 0], cov[0, 1], cov[1, 1])
    ang = 0.5 * math.atan2(2.0 * cov[0, 1], cov[0, 0] - cov[1, 1])
    ex = np.sqrt(max(l1, 1e-9)) * np.cos(t)
    ey = np.sqrt(max(l2, 1e-9)) * np.sin(t)
    elx = ex * math.cos(ang) - ey * math.sin(ang)
    ely = ex * math.sin(ang) + ey * math.cos(ang)

    fig_points = go.Figure()
    fig_points.add_trace(go.Scatter(x=xs, y=ys, mode='markers+text', text=[f'x{i}' for i in range(1, len(pts) + 1)], textposition='top center', marker=dict(size=10, color='#2563eb'), name='points'))
    fig_points.add_trace(go.Scatter(x=[0], y=[0], mode='markers', marker=dict(size=14, color='#dc2626', symbol='x'), name='query'))
    fig_points.add_trace(go.Scatter(x=elx, y=ely, mode='lines', line=dict(dash='dash', color='black'), name='1-sigma'))
    fig_points.update_layout(title='Sigma effect on distances', height=460, margin=dict(l=50, r=20, t=45, b=45))
    fig_points.update_xaxes(scaleanchor='y')

    fig_table = go.Figure(data=[go.Table(
        header=dict(values=['Point', 'Euclidean', 'Mahalanobis']),
        cells=dict(values=[[f'x{i}' for i in range(1, len(pts) + 1)], [f'{v:.3f}' for v in de], [f'{v:.3f}' for v in dm]])
    )])
    fig_table.update_layout(title='Distance table', height=460, margin=dict(l=10, r=10, t=45, b=10))

    with out:
        out.clear_output(wait=True)
        display(widgets.HBox([fig_points, fig_table]))
        print('Query point fixed at (0,0). Move the four points with sliders and adjust Sigma to compare Euclidean and Mahalanobis distances.')


for xw, yw in point_widgets:
    xw.observe(render, names='value')
    yw.observe(render, names='value')
for w in [sx, sy, rho]:
    w.observe(render, names='value')


def on_reset(_):
    for (xw, yw), (xv, yv) in zip(point_widgets, DEFAULT_POINTS):
        xw.value = xv
        yw.value = yv


reset_btn.on_click(on_reset)

display(widgets.VBox([
    widgets.HTML(value='<h2 style="margin:0 0 8px;">Interactive Sigma Control (Lecture 10)</h2><div style="color:#475569;margin-bottom:10px;">Move the four points with sliders and vary covariance Sigma for Euclidean vs Mahalanobis comparison.</div>'),
    widgets.HBox(point_boxes, layout=widgets.Layout(flex_flow='row wrap', gap='12px')),
    widgets.VBox([sx, sy, rho], layout=widgets.Layout(border='1px solid #cbd5e1', padding='10px', border_radius='12px', width='320px')),
    reset_btn,
    out,
], layout=widgets.Layout(gap='16px')))
render()


# Lecture 10: Editable Points and Covariance $\Sigma$

- Edit a small set of points (x1,y1,...).
- Observe Euclidean vs Mahalanobis distances from a query point.
- Control covariance with $(\sigma_x, \sigma_y, \rho)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def parse_points(txt):
    pts = []
    for row in txt.strip().split(';'):
        row=row.strip()
        if not row:
            continue
        a,b = row.split(',')
        pts.append([float(a), float(b)])
    arr=np.array(pts, dtype=float)
    if arr.ndim!=2 or arr.shape[1]!=2 or len(arr)==0:
        raise ValueError('Expected format: x1,y1; x2,y2; ...')
    return arr


def draw(points_text='0,0; 1,1; 2,0.5; -1,1.6; -2,-0.8', qx=0.0, qy=0.0, sigma_x=1.0, sigma_y=1.0, rho=0.0):
    pts = parse_points(points_text)
    q = np.array([qx, qy])

    cov = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                    [rho*sigma_x*sigma_y, sigma_y**2]])
    inv = np.linalg.pinv(cov)

    dif = pts - q
    d_e = np.sqrt((dif**2).sum(axis=1))
    d_m = np.sqrt(np.sum((dif @ inv) * dif, axis=1))

    fig, ax = plt.subplots(figsize=(6.5,6))
    ax.scatter(pts[:,0], pts[:,1], c='#2563eb', s=45, label='data points')
    ax.scatter([q[0]], [q[1]], c='#dc2626', s=90, marker='x', label='query')

    # covariance ellipse
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2*np.pi, 240)
    unit = np.vstack([np.cos(t), np.sin(t)])
    ell = vecs @ (np.sqrt(vals)[:,None] * unit)
    ax.plot(q[0] + ell[0], q[1] + ell[1], 'k--', lw=1.5, label='1-sigma ellipse')

    for i,p in enumerate(pts):
        ax.plot([q[0], p[0]], [q[1], p[1]], color='gray', alpha=0.4)
        ax.text(p[0]+0.06, p[1]+0.06, f'{i+1}: E={d_e[i]:.2f}, M={d_m[i]:.2f}', fontsize=8)

    ax.set_title('Point distances with covariance-aware metric')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    plt.show()

    order_e = np.argsort(d_e)
    order_m = np.argsort(d_m)
    print('Nearest by Euclidean:', [int(i+1) for i in order_e[:3]])
    print('Nearest by Mahalanobis:', [int(i+1) for i in order_m[:3]])

widgets.interact(
    draw,
    points_text=widgets.Text(description='points', value='0,0; 1,1; 2,0.5; -1,1.6; -2,-0.8', layout=widgets.Layout(width='850px')),
    qx=widgets.FloatSlider(description='q_x', min=-3, max=3, step=0.1, value=0.0, continuous_update=False),
    qy=widgets.FloatSlider(description='q_y', min=-3, max=3, step=0.1, value=0.0, continuous_update=False),
    sigma_x=widgets.FloatSlider(description='sigma_x', min=0.2, max=3, step=0.1, value=1.0, continuous_update=False),
    sigma_y=widgets.FloatSlider(description='sigma_y', min=0.2, max=3, step=0.1, value=1.0, continuous_update=False),
    rho=widgets.FloatSlider(description='rho', min=-0.95, max=0.95, step=0.05, value=0.0, continuous_update=False),
)
